# NVIDIA (NVDA) Stock Exploratory Data Analysis
## A Senior Data Scientist's Perspective

This notebook explores the historical price action of NVDA, focusing on statistical properties, distribution, and volatility. Our goal is to understand the underlying data generating process before moving into modeling.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

### 1. Data Loading
We pull the data from our DuckDB warehouse.

In [ ]:
con = duckdb.connect("stocks.db")
df = con.execute("SELECT * FROM stock_prices WHERE ticker = 'NVDA' ORDER BY date").df()
con.close()

df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df.head()

### 2. Time Series Visualization
Visualizing the raw closing prices and volume.

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(df.index, df['close'], label='Close Price', color='tab:blue')
ax1.set_ylabel('Price ($)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_title('NVDA Historical Price and Volume')

ax2 = ax1.twinx()
ax2.fill_between(df.index, df['volume'], alpha=0.3, color='gray', label='Volume')
ax2.set_ylabel('Volume', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')
ax2.grid(False)

plt.show()

### 3. Summary Statistics & Distribution
Financial time series are rarely normally distributed. We look for 'Fat Tails' and skewness.

In [ ]:
# Calculate Daily Returns
df['returns'] = df['close'].pct_change().dropna()

print("--- Price Summary ---")
print(df['close'].describe())

print("\n--- Returns Summary ---")
print(df['returns'].describe())

print(f"\nSkewness: {df['returns'].skew():.4f}")
print(f"Kurtosis: {df['returns'].kurtosis():.4f}") # High kurtosis (>3) indicates fat tails

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['returns'].dropna(), kde=True, bins=100)
plt.title('Distribution of Daily Returns (Log Scale on Y for tail clarity)')
plt.yscale('log')
plt.show()

### 4. Volatility Analysis
Using a rolling standard deviation to capture regime changes in volatility.

In [ ]:
df['volatility'] = df['returns'].rolling(window=21).std() * np.sqrt(252) # Annualized 21-day vol
df['volatility'].plot(title='21-Day Annualized Volatility')
plt.ylabel('Annualized Vol')
plt.show()